# NB08 — Skenario C: FQA + Enhancement + Re-clustering

**Kontribusi utama skripsi**: pipeline penuh FQA → Adaptive Enhancement → ArcFace
embedding baru → HDBSCAN re-clustering → perbandingan dengan Skenario A (baseline).

**Input:**
- `output_nb05/metadata_labeled.pkl` — metadata + bbox + kps (jika ada)
- `output_nb07/skenario_b_results.pkl` — threshold θ dari Youden's J
- `output_nb06/skenario_a_results.pkl` — baseline untuk tabel perbandingan
- `output_labeling/face_labels_verified.csv` — ground-truth
- Foto asli di `DOKUMENTASI OUTDOOR 2025/`

**Output:** `output_nb08/embeddings_enhanced.npy`, `skenario_c_results.pkl`,
`perbandingan_a_vs_c.csv`, `perbandingan_a_vs_c.png`

**Runtime:** GPU wajib (untuk ArcFace re-embedding ~15k wajah)

In [ ]:
# Cell 1 — Install
!pip install -q insightface onnxruntime-gpu hdbscan scikit-learn
try:
    import faiss
    print("faiss OK:", faiss.__version__)
except ImportError:
    !pip install -q faiss-gpu
    import faiss

In [ ]:
# Cell 2 — Imports & Mount
import os, pickle, time, math, warnings
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt

import insightface
from insightface.app import FaceAnalysis
from insightface.utils import face_align

import hdbscan
from hdbscan.validity import validity_index
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix,
)

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

In [ ]:
# Cell 3 — Config
BASE       = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB05_DIR   = BASE / "output_nb05"
NB06_DIR   = BASE / "output_nb06"
NB07_DIR   = BASE / "output_nb07"
LABEL_DIR  = BASE / "output_labeling"
OUTPUT_DIR = BASE / "output_nb08"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Konstanta FQA — identik dengan NB07 agar hasil reprodusibel
VARLAP_REF   = 300.0
NOISE_REF    = 15.0
ILLUM_SIGMA  = 50.0
CONTRAST_REF = 60.0
ALIGN_SIZE   = 512
ARCFACE_SIZE = 112

# HDBSCAN — params IDENTIK dengan NB05 untuk perbandingan adil (satu variabel berubah)
HDBSCAN_MCS = 50
HDBSCAN_MS  = 5

SUBSAMPLE   = 2000   # untuk Silhouette & DBCV (sama dengan NB06)
RANDOM_SEED = 42

print('Config OK.')
print(f'  NB05 : {NB05_DIR}')
print(f'  NB06 : {NB06_DIR}')
print(f'  NB07 : {NB07_DIR}')
print(f'  Out  : {OUTPUT_DIR}')

In [ ]:
# Cell 4 — Fungsi FQA + Enhancement
# Copy-paste PERSIS dari EXP_Classical_vs_DL_Restoration.ipynb
# untuk memastikan reprodusibilitas (konstanta, formula, urutan eksekusi)

# ── FQA ──────────────────────────────────────────────────────────────────

def immerkaer_sigma(gray):
    """Fast noise variance estimator (Immerkær, 1996)."""
    H, W = gray.shape
    if H < 2 or W < 2:
        return 0.0
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float64)
    resp = cv2.filter2D(gray.astype(np.float64), -1, kernel)
    return np.sqrt(np.abs(resp).sum() * math.pi / (6 * (H - 2) * (W - 2)) + 1e-9)


def compute_sub_scores(native_bgr, bbox, kps):
    """Hitung FQA sub-scores dari crop wajah native resolution.
    PENTING: dihitung SEBELUM alignment/resize untuk menghindari bias resolusi.
    """
    gray = cv2.cvtColor(native_bgr, cv2.COLOR_BGR2GRAY)

    var_lap = cv2.Laplacian(gray, cv2.CV_64F).var()
    blur_s  = float(np.clip(var_lap / VARLAP_REF, 0, 1))

    bw    = bbox[2] - bbox[0]
    bh    = bbox[3] - bbox[1]
    res_s = float(np.clip(min(bw, bh) / 112.0, 0, 1))

    mean_b       = float(gray.mean())
    illum_mean_s = float(math.exp(-((mean_b - 128.0) ** 2) / (2 * ILLUM_SIGMA ** 2)))
    contrast_s   = float(np.clip(gray.std() / CONTRAST_REF, 0, 1))

    sigma   = immerkaer_sigma(gray)
    noise_s = float(np.clip(1.0 - sigma / NOISE_REF, 0, 1))

    if kps is not None:
        le, re, nose = kps[0], kps[1], kps[2]
        d_l = abs(nose[0] - le[0])
        d_r = abs(re[0] - nose[0])
        yaw = min(d_l, d_r) / max(d_l, d_r) if max(d_l, d_r) > 0 else 0.0
    else:
        yaw = None

    return {
        "blur":        blur_s,
        "resolution":  res_s,
        "illumination": 0.5 * illum_mean_s + 0.5 * contrast_s,
        "noise":       noise_s,
        "pose":        yaw,
        "_mean_brightness_raw": mean_b,
        "_contrast_score":      contrast_s,
        "_illum_mean_s":        illum_mean_s,
    }


# ── Alignment (SR = cubic warp) ───────────────────────────────────────────

def warp512(img_full, kps, cubic=False):
    """Warp wajah ke kanvas ALIGN_SIZE×ALIGN_SIZE menggunakan landmark InsightFace.
    cubic=True → Bicubic SR (resolusi lebih baik untuk wajah kecil).
    """
    M    = face_align.estimate_norm(kps, image_size=ALIGN_SIZE)
    flag = cv2.INTER_CUBIC if cubic else cv2.INTER_LINEAR
    return cv2.warpAffine(img_full, M, (ALIGN_SIZE, ALIGN_SIZE),
                          flags=flag, borderValue=0.0)


# ── Enhancement functions (dari EXP notebook, urutan wajib: NLM→Gamma→CLAHE→Sharpen) ─

def apply_nlm_denoising(img):
    return cv2.fastNlMeansDenoisingColored(img, None, 5, 5, 7, 21)


def apply_gamma_correction(img, mean_raw):
    m     = float(np.clip(mean_raw / 255.0, 1e-3, 0.999))
    gamma = float(np.clip(math.log(0.5) / math.log(m), 0.4, 2.5))
    lut   = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                       for i in range(256)]).astype("uint8")
    return cv2.LUT(img, lut)


def apply_clahe(img):
    lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


def apply_laplacian_sharpening(img):
    k = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    return cv2.filter2D(img, -1, k)


def route_enhancement(aligned512, fqa, th):
    """Terapkan chain enhancement berdasarkan FQA sub-scores vs threshold θ.
    Return (img_enhanced, list_teknik_yang_diterapkan).
    
    Catatan: θ_illumination dari NB07 (skor gabungan) dipakai untuk kedua gate
    _illum_mean_s DAN _contrast_score (aproksimasi, documented sebagai keterbatasan).
    """
    img     = aligned512.copy()
    applied = []

    if fqa["noise"] < th.get("noise", 0.6):
        img = apply_nlm_denoising(img)
        applied.append("denoising")

    if fqa["_illum_mean_s"] < th.get("illumination", 0.7):
        img = apply_gamma_correction(img, fqa["_mean_brightness_raw"])
        applied.append("gamma")

    if fqa["_contrast_score"] < th.get("illumination", 0.7):
        img = apply_clahe(img)
        applied.append("clahe")

    if fqa["blur"] < th.get("blur", 0.6):
        img = apply_laplacian_sharpening(img)
        applied.append("sharpen")

    return img, applied


# ── ArcFace embedding ─────────────────────────────────────────────────────

def embed(rec_model, crop512_bgr):
    """Resize 512→112 (INTER_AREA) dan ekstrak ArcFace embedding.
    Output: float32 512-dim, L2-normalized.
    """
    crop112 = cv2.resize(crop512_bgr, (ARCFACE_SIZE, ARCFACE_SIZE),
                         interpolation=cv2.INTER_AREA)
    emb = rec_model.get_feat(crop112).flatten().astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-9)


print('Fungsi FQA + Enhancement + ArcFace terdefinisi.')

In [ ]:
# Cell 5 — Load Model InsightFace + Threshold dari NB07

# InsightFace buffalo_l (sama dengan NB01 untuk konsistensi)
app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec_model = app.models["recognition"]
print(f'InsightFace loaded. Providers aktif: {rec_model.session.get_providers()}')

# Load threshold dari Skenario B (Youden's J)
with open(NB07_DIR / "skenario_b_results.pkl", "rb") as f:
    nb07 = pickle.load(f)

th = {sc: float(v["theta"]) for sc, v in nb07["thresholds"].items()}

print("\nThreshold θ dari NB07 (Youden's J):")
for sc, theta in th.items():
    print(f"  {sc:15s} θ = {theta:.3f}")
print()

# Fallback jika 'resolution' tidak ada di NB07 (ROC tidak menghasilkan pemisahan baik)
if "resolution" not in th:
    th["resolution"] = 0.7
    print("⚠  'resolution' tidak ada di NB07 → pakai fallback θ=0.7 (CLAUDE.md placeholder)")

In [ ]:
# Cell 6 — Load Metadata + Ground-truth

with open(NB05_DIR / "metadata_labeled.pkl", "rb") as f:
    meta = pickle.load(f)
N = len(meta)

df_gt = pd.read_csv(LABEL_DIR / "face_labels_verified.csv")
assert len(df_gt) == N, f"Mismatch: {len(df_gt)} ground-truth vs {N} meta"

has_kps = "kps" in meta[0]

print(f"Total wajah  : {N:,}")
print(f"kps tersedia : {'YA (digunakan langsung)' if has_kps else 'TIDAK (re-detect dengan InsightFace)'}")
print(f"Ground-truth : {len(df_gt):,} baris, {df_gt['identity_id'].max()+1} identitas unik")

In [ ]:
# Cell 7 — Loop Utama: FQA → Enhancement → ArcFace Embedding
# Baca setiap foto 1× saja (photo caching), proses semua wajah di foto itu.
# Jika kps tidak ada di metadata → re-detect menggunakan InsightFace app.get()

# Kelompokkan global index berdasarkan photo_path
photo_to_indices = defaultdict(list)
for gi, m in enumerate(meta):
    photo_to_indices[m["photo_path"]].append(gi)

new_embeddings = np.zeros((N, 512), dtype=np.float32)
n_enhanced = 0
n_sr       = 0        # jumlah wajah yang kena Bicubic SR
n_failed   = 0
tech_count = Counter()  # statistik teknik enhancement

t_start = time.time()
print(f"Memproses {len(photo_to_indices):,} foto ({N:,} wajah)...")

for idx_ph, (photo_path, indices) in enumerate(photo_to_indices.items()):
    if idx_ph % 50 == 0:
        elapsed = time.time() - t_start
        done    = sum(len(v) for k, v in list(photo_to_indices.items())[:idx_ph])
        print(f"  [{idx_ph:4d}/{len(photo_to_indices)}] wajah={done:6d}/{N}  "
              f"enhanced={n_enhanced}  t={elapsed:.0f}s", end="\r")

    img = cv2.imread(str(photo_path))
    if img is None:
        n_failed += len(indices)
        continue

    # Re-detect jika kps tidak ada di metadata
    detected_map = None
    if not has_kps:
        detected_faces = app.get(img)
        detected_map   = {tuple(int(v) for v in f.bbox): f for f in detected_faces}

    for gi in indices:
        m    = meta[gi]
        bbox = m["bbox"]
        x1, y1, x2, y2 = [int(v) for v in bbox]

        # Native crop untuk FQA (sebelum alignment/resize — KRITIS, baca CLAUDE.md)
        native_crop = img[max(0, y1):y2, max(0, x1):x2]
        if native_crop.size == 0:
            n_failed += 1
            continue

        # Resolve kps
        kps = m.get("kps", None)
        if kps is None and detected_map is not None:
            key = min(
                detected_map.keys(),
                key=lambda b: abs(b[0]-x1)+abs(b[1]-y1)+abs(b[2]-x2)+abs(b[3]-y2),
                default=None,
            )
            if key:
                kps = detected_map[key].kps

        if kps is None:
            n_failed += 1
            continue

        kps = np.array(kps)

        # FQA — dihitung pada native crop (bukan setelah alignment)
        fqa = compute_sub_scores(native_crop, bbox, kps)

        # SR: alignment ke 512×512 (cubic jika resolusi rendah)
        use_cubic = fqa["resolution"] < th.get("resolution", 0.7)
        aligned   = warp512(img, kps, cubic=use_cubic)
        if use_cubic:
            n_sr += 1

        # Enhancement chain (urutan wajib dari CLAUDE.md: NLM→Gamma→CLAHE→Sharpen)
        enhanced, applied = route_enhancement(aligned, fqa, th)
        if applied:
            n_enhanced += 1
            for t in applied:
                tech_count[t] += 1

        # ArcFace embedding dari enhanced crop
        new_embeddings[gi] = embed(rec_model, enhanced)

elapsed_total = time.time() - t_start
print(f"\nSelesai dalam {elapsed_total:.1f}s")
print(f"  Enhanced (≥1 teknik): {n_enhanced:,} / {N:,} ({n_enhanced/N*100:.1f}%)")
print(f"  Bicubic SR           : {n_sr:,} wajah")
print(f"  Gagal/skip           : {n_failed:,} wajah")
print(f"  Teknik per frekuensi : {dict(tech_count.most_common())}")

np.save(OUTPUT_DIR / "embeddings_enhanced.npy", new_embeddings)
print(f"\nTersimpan: output_nb08/embeddings_enhanced.npy  shape={new_embeddings.shape}")

In [ ]:
# Cell 8 — HDBSCAN Re-clustering
# Params IDENTIK dengan NB05 — satu variabel yang berubah hanyalah embeddings

print(f"HDBSCAN mcs={HDBSCAN_MCS}, ms={HDBSCAN_MS}, method=eom, metric=euclidean")
t0 = time.time()

clusterer_c = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MCS,
    min_samples=HDBSCAN_MS,
    cluster_selection_method="eom",
    metric="euclidean",
    core_dist_n_jobs=-1,
)

# Masking: wajah yang gagal (embedding = zero) dikecualikan agar tidak jadi outlier buatan
zero_mask  = np.linalg.norm(new_embeddings, axis=1) < 1e-6
valid_mask = ~zero_mask
print(f"Wajah zero-embedding (failed): {zero_mask.sum()}")

labels_c_valid = clusterer_c.fit_predict(new_embeddings[valid_mask])

# Rekonstruksi array label full (wajah failed = -1 noise)
labels_c = np.full(N, -1, dtype=np.int32)
labels_c[valid_mask] = labels_c_valid

n_clusters_c = int(labels_c.max() + 1) if labels_c.max() >= 0 else 0
noise_pct_c  = float((labels_c == -1).mean() * 100)
coverage_c   = float((labels_c >= 0).mean() * 100)
n_noise_c    = int((labels_c == -1).sum())
n_clustered_c = int((labels_c >= 0).sum())

sizes_c = sorted(Counter(labels_c[labels_c >= 0].tolist()).values(), reverse=True)

print(f"\nSkenario C — HDBSCAN hasil:")
print(f"  Cluster   : {n_clusters_c}")
print(f"  Ter-cluster: {n_clustered_c:,} ({coverage_c:.2f}%)")
print(f"  Noise     : {n_noise_c:,} ({noise_pct_c:.2f}%)")
print(f"  Size: min={min(sizes_c)}, max={max(sizes_c)}, mean={np.mean(sizes_c):.1f}")
print(f"  Waktu HDBSCAN: {time.time()-t0:.1f}s")

In [ ]:
# Cell 9 — Metrik Internal (Silhouette, DBCV, DBI)
# Metodologi identik dengan NB06 untuk komparabilitas

D = new_embeddings.shape[1]

# Subsample dari wajah yang ter-cluster
rng      = np.random.default_rng(seed=RANDOM_SEED)
idx_pool = np.where(labels_c >= 0)[0]
n_sub    = min(SUBSAMPLE, len(idx_pool))
idx_sub  = np.sort(rng.choice(idx_pool, size=n_sub, replace=False))

emb_sub = new_embeddings[idx_sub].copy().astype(np.float32)
faiss.normalize_L2(emb_sub)   # cosine via inner product

# ── Silhouette (cosine distance, precomputed via FAISS) ──────────────────
print(f"Menghitung Silhouette (n_sub={n_sub})...")
t0 = time.time()
try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
    print("  FAISS GPU aktif.")
except Exception:
    index = faiss.IndexFlatIP(D)
    print("  FAISS CPU (GPU tidak tersedia).")

index.add(emb_sub)
sim_matrix, _ = index.search(emb_sub, n_sub)
dist_matrix   = np.clip(1.0 - sim_matrix.astype(np.float64), 0.0, 2.0)
np.fill_diagonal(dist_matrix, 0.0)

labels_sub = labels_c[idx_sub]
sil_score_c = silhouette_score(dist_matrix, labels_sub, metric='precomputed')
print(f"  Silhouette = {sil_score_c:.4f}  ({time.time()-t0:.1f}s)")

# ── DBCV ─────────────────────────────────────────────────────────────────
print("Menghitung DBCV...")
t0 = time.time()
mask_valid   = labels_sub >= 0
dist_sub     = dist_matrix[np.ix_(mask_valid, mask_valid)]
labels_valid = labels_sub[mask_valid]

label_counts   = Counter(labels_valid.tolist())
valid_clusters = {lbl for lbl, cnt in label_counts.items() if cnt >= 3}
keep           = np.isin(labels_valid, list(valid_clusters))
dist_sub_v     = dist_sub[np.ix_(keep, keep)]
labels_v       = labels_valid[keep]

unique_labels = np.unique(labels_v)
remap         = {old: new for new, old in enumerate(unique_labels)}
labels_v_remap = np.array([remap[l] for l in labels_v])

try:
    dbcv_score_c = validity_index(dist_sub_v, labels_v_remap, metric='precomputed', d=D)
    print(f"  DBCV = {dbcv_score_c:.4f}  ({time.time()-t0:.1f}s)")
except AssertionError as e:
    dbcv_score_c = float('nan')
    print(f"  DBCV gagal (hdbscan bug): {e}")

# ── DBI ───────────────────────────────────────────────────────────────────
print("Menghitung DBI...")
t0 = time.time()
emb_sub_f64 = emb_sub[mask_valid].astype(np.float64)
dbi_score_c = davies_bouldin_score(emb_sub_f64, labels_valid)
print(f"  DBI = {dbi_score_c:.4f}  ({time.time()-t0:.1f}s)")

In [ ]:
# Cell 10 — Metrik Eksternal (Purity, ARI, NMI)
# Metodologi identik dengan NB06

def purity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return float(cm.max(axis=0).sum() / cm.sum())

# Mapping global_index → label cluster baru (Skenario C)
df_gt["cluster_c"] = df_gt["global_index"].map(dict(zip(range(N), labels_c)))

# Hanya pada wajah terverifikasi (identity_id >= 0)
df_eval = df_gt[df_gt["identity_id"] >= 0].copy()
y_true  = df_eval["identity_id"].values
y_pred  = df_eval["cluster_c"].values

print(f"Wajah untuk evaluasi eksternal: {len(df_eval):,}")
print(f"Identitas unik (ground-truth) : {len(np.unique(y_true))}")
print(f"Cluster unik (Skenario C pred): {len(np.unique(y_pred))}")

purity_c = purity_score(y_true, y_pred)
ari_c    = adjusted_rand_score(y_true, y_pred)
nmi_c    = normalized_mutual_info_score(y_true, y_pred, average_method='arithmetic')

print(f"\nPurity = {purity_c:.4f}")
print(f"ARI    = {ari_c:.4f}")
print(f"NMI    = {nmi_c:.4f}")

In [ ]:
# Cell 11 — Tabel Perbandingan Skenario A vs Skenario C

with open(NB06_DIR / "skenario_a_results.pkl", "rb") as f:
    sa = pickle.load(f)

metrics_internal_c = {
    "coverage_pct" : round(coverage_c, 4),
    "noise_pct"    : round(noise_pct_c, 4),
    "n_clusters"   : n_clusters_c,
    "silhouette"   : round(float(sil_score_c), 4),
    "dbcv"         : round(float(dbcv_score_c), 4) if not np.isnan(dbcv_score_c) else None,
    "dbi"          : round(float(dbi_score_c), 4),
}
metrics_external_c = {
    "purity" : round(purity_c, 4),
    "ari"    : round(ari_c, 4),
    "nmi"    : round(nmi_c, 4),
}

row_a = {"Skenario": "A — Baseline (tanpa enhancement)",
         **sa["metrics_internal"], **sa["metrics_external"]}
row_c = {"Skenario": "C — FQA + Enhancement",
         **metrics_internal_c, **metrics_external_c}

df_compare = pd.DataFrame([row_a, row_c])

print("=" * 70)
print("  PERBANDINGAN SKENARIO A vs C")
print("=" * 70)
display(df_compare.set_index("Skenario").T)

# Delta: C minus A (positif = C lebih baik)
numeric_cols = ["coverage_pct", "silhouette", "dbcv", "purity", "ari", "nmi"]
lower_better = ["noise_pct", "dbi"]  # lebih kecil = lebih baik

print("\nDelta C vs A (+ = C lebih baik):")
for col in [c for c in df_compare.columns if c != "Skenario" and col != "n_clusters"]:
    val_a = df_compare.loc[0, col]
    val_c = df_compare.loc[1, col]
    if val_a is None or val_c is None:
        print(f"  {col:20s}: NaN (DBCV gagal, tidak bisa dibandingkan)")
        continue
    delta = val_c - val_a
    good  = delta > 0 if col not in lower_better else delta < 0
    mark  = "✓" if good else "✗"
    print(f"  {col:20s}: {delta:+.4f}  {mark}")

In [ ]:
# Cell 12 — Visualisasi: Bar Chart A vs C

metrics_to_plot = [
    ("silhouette",   "Silhouette",   True,  "Lebih tinggi = lebih baik"),
    ("dbi",          "DBI",          False, "Lebih rendah = lebih baik"),
    ("coverage_pct", "Coverage (%)", True,  "Lebih tinggi = lebih baik"),
    ("purity",       "Purity",       True,  "Lebih tinggi = lebih baik"),
    ("ari",          "ARI",          True,  "Lebih tinggi = lebih baik"),
    ("nmi",          "NMI",          True,  "Lebih tinggi = lebih baik"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

colors_a = "#4472C4"
colors_c = "#ED7D31"

for ax, (key, label, higher_better, note) in zip(axes, metrics_to_plot):
    val_a = sa["metrics_internal"].get(key) or sa["metrics_external"].get(key)
    val_c = metrics_internal_c.get(key) or metrics_external_c.get(key)

    bars = ax.bar(["Skenario A\n(Baseline)", "Skenario C\n(FQA+Enh)"],
                  [val_a if val_a is not None else 0,
                   val_c if val_c is not None else 0],
                  color=[colors_a, colors_c], width=0.5)

    for bar, val in zip(bars, [val_a, val_c]):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{val:.4f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_title(f"{label}\n({note})", fontsize=9)
    ax.set_ylabel(label, fontsize=8)

    if val_a is not None and val_c is not None:
        delta = val_c - val_a
        good  = delta > 0 if higher_better else delta < 0
        marker = f"Δ = {delta:+.4f} {'✓' if good else '✗'}"
        ax.set_xlabel(marker, fontsize=8,
                      color='green' if good else 'red')

plt.suptitle("Perbandingan Skenario A vs C — FQA + Adaptive Enhancement",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "perbandingan_a_vs_c.png", dpi=150, bbox_inches='tight')
plt.show()
print("Tersimpan: output_nb08/perbandingan_a_vs_c.png")

In [ ]:
# Cell 13 — Simpan Output

skenario_c = {
    "skenario"    : "C",
    "deskripsi"   : "FQA + Adaptive Enhancement + HDBSCAN re-clustering (mcs=50, ms=5)",
    "dataset"     : {
        "n_total"       : N,
        "n_clustered"   : n_clustered_c,
        "n_noise"       : n_noise_c,
        "n_enhanced"    : n_enhanced,
        "n_sr"          : n_sr,
        "n_failed"      : n_failed,
        "n_verified"    : int(len(df_eval)),
        "n_identities"  : int(len(np.unique(y_true))),
        "tech_count"    : dict(tech_count),
    },
    "params"      : {
        "min_cluster_size"        : HDBSCAN_MCS,
        "min_samples"             : HDBSCAN_MS,
        "cluster_selection_method": "eom",
        "metric"                  : "euclidean",
    },
    "thresholds_used" : th,
    "fqa_constants"   : {
        "VARLAP_REF": VARLAP_REF,
        "NOISE_REF" : NOISE_REF,
        "ILLUM_SIGMA": ILLUM_SIGMA,
        "CONTRAST_REF": CONTRAST_REF,
    },
    "metrics_internal" : metrics_internal_c,
    "metrics_external" : metrics_external_c,
}

with open(OUTPUT_DIR / "skenario_c_results.pkl", "wb") as f:
    pickle.dump(skenario_c, f)

df_compare.to_csv(OUTPUT_DIR / "perbandingan_a_vs_c.csv", index=False)

print("Output tersimpan di:", OUTPUT_DIR)
print("  ✓ embeddings_enhanced.npy   (15k × 512, embedding baru pasca-enhancement)")
print("  ✓ skenario_c_results.pkl    (dict lengkap semua metrik)")
print("  ✓ perbandingan_a_vs_c.csv   (tabel 2 baris A vs C)")
print("  ✓ perbandingan_a_vs_c.png   (bar chart A vs C)")
print()
print("=" * 60)
print("  RINGKASAN AKHIR SKENARIO C")
print("=" * 60)
print(f"  Wajah dienhance  : {n_enhanced:,} / {N:,} ({n_enhanced/N*100:.1f}%)")
print(f"  Bicubic SR       : {n_sr:,} wajah")
print(f"  Cluster          : {n_clusters_c}")
print(f"  Coverage         : {coverage_c:.2f}%")
print(f"  Noise rate       : {noise_pct_c:.2f}%")
print(f"  Silhouette       : {sil_score_c:.4f}")
print(f"  DBCV             : {dbcv_score_c:.4f}" if not np.isnan(dbcv_score_c) else "  DBCV             : NaN")
print(f"  DBI              : {dbi_score_c:.4f}")
print(f"  Purity           : {purity_c:.4f}")
print(f"  ARI              : {ari_c:.4f}")
print(f"  NMI              : {nmi_c:.4f}")
print("=" * 60)